# Ablation Experiments

This notebook runs targeted ablation experiments to investigate:
1. **Continuation ablation** — disable `continuation=True` so all methods reset weights per lambda stage
2. **Single lambda** — run only lambda=0.5 to avoid cross-stage effects
3. **Alternative fairness measure** — compare MAD vs Gap vs Atkinson
4. **Predictor architecture** — linear, MLP, wider MLP, ResNet-Tabular, FT-Transformer
5. **Learning rate × steps** — convergence sensitivity grid
6. **Optimizer** — SGD vs Adam
7. **Gradient cosine table** — print per-iteration gradient diagnostics

### Data split: 50% train / 50% test (no validation)

### Training note: Full-batch SGD
All methods use **full-batch gradient descent** (`batch_size=-1` in `DEFAULT_TRAIN_CFG`).
This means every gradient step uses the entire training set — there is no mini-batch stochasticity.
The optimizer is SGD (default) or Adam, applied identically to all methods.
The only difference between methods is **which gradients are combined** and **how**.

## Experiment Design Notes

### Why FDFL starts with positive cos(g_dec, g_fair) while other methods start negative

In the gradient cosine similarity plots at lambda=0.2, FDFL shows a positive
cos(g_dec, g_fair) at iteration 0 (~0.8), while WS-equal, WS-dec, MGDA, PCGrad,
and FFO all start negative (~-0.3). This is **not a bug** — it is a direct
consequence of the **continuation** setting.

Most methods (WS-equal, WS-dec, MGDA, PCGrad, FFO) use `continuation=True`:
model weights carry over across lambda stages (0.0 -> 0.5, with 70 training steps each). FDFL is the exception with `continuation=False`, meaning
it resets to random initialization at every lambda stage.

When we plot at lambda=0.5 (the 2nd stage):
- **Continuation methods** enter this stage having already trained for 70 steps
  (at lambda=0.0). The model is near the Pareto frontier
  where improving decisions further comes at the cost of group equity — hence
  g_dec and g_fair **conflict** (negative cosine).
- **FDFL** starts lambda=0.2 from fresh random initialization. At random weights,
  both "improve decisions" and "equalize across groups" push predictions in similar
  directions away from noise — hence they **align** (positive cosine). The conflict
  only emerges after several training steps.

This notebook's Experiment 1 confirms this: when we disable continuation for all
methods, they all start with similar positive cosine at each lambda stage.

### The continuation computational shortcut

Continuation (`continuation=True`) serves two purposes:
1. **Warm-starting**: the model does not waste steps re-learning basic prediction
   structure at higher lambda values.
2. **Pareto frontier tracing**: by gradually increasing the fairness penalty lambda,
   we trace out points along the Pareto frontier. Each stage refines the previous
   solution rather than solving from scratch.

The trade-off is path-dependence: the lambda=0.5 result depends on the full
training history (0.0 -> 0.5). Experiments 1 and 2 in this
notebook isolate methods from this effect.

### Other implementation details

- **FPTO warmstart** (`warmstart_fraction=0.25`): PLG and FPLG run the first 25%
  of steps as pure FPTO (prediction + fairness only, no decision gradient). This
  stabilizes the predictor before introducing the noisier decision gradient.
- **Alpha schedule** (inv_sqrt decay, alpha0=1.0): the prediction gradient weight
  decays as alpha(t) = 1/sqrt(t+1). Early training is prediction-dominated; later
  steps focus on decision quality.
- **Full-batch training** (`batch_size=-1`): every step uses the entire training set.
  Required because the resource allocation solver needs all patients simultaneously
  to respect the global budget constraint. Also eliminates mini-batch noise from
  gradient cosine measurements.
- **Gradient clipping** (`grad_clip_norm=10000`): safety net against exploding
  gradients, set high enough that it rarely activates.
- **Fairness smoothing** (`fairness_smoothing=1e-6`): epsilon in fairness metric
  denominators to avoid division by zero.
- **force_lambda_path_all_methods=True**: even methods without a fairness objective
  run through all lambda stages, ensuring every method produces results at every
  lambda for fair comparison.

In [ ]:
# Cell 1: Install dependencies (Colab only)
!pip install -q torch numpy pandas scipy matplotlib cvxpy mosek
!pip install -q git+https://github.com/cvxpy/cvxtorch.git
!pip install -q --upgrade threadpoolctl

In [ ]:
# Cell 2: Path setup
import os, sys, copy, importlib
import numpy as np
import pandas as pd

# --- Colab: mount drive ---
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/colab_upload'

# Verify path exists
if not os.path.exists(DRIVE_ROOT):
    print(f'ERROR: {DRIVE_ROOT} not found. Please check the path.')

MOSEK_LIC_PATH = os.path.join(DRIVE_ROOT, 'data', 'mosek.lic')
if os.path.exists(MOSEK_LIC_PATH):
    os.environ['MOSEKLM_LICENSE_FILE'] = MOSEK_LIC_PATH
    print(f'MOSEK license set: {MOSEK_LIC_PATH}')

# Add DRIVE_ROOT and src to sys.path
for p in [DRIVE_ROOT, os.path.join(DRIVE_ROOT, 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Force reload all fair_dfl modules to pick up any Drive file changes
mods_to_remove = [k for k in sys.modules if k.startswith('fair_dfl') or k in ('configs', 'plotting', 'analysis')]
for k in mods_to_remove:
    del sys.modules[k]

# CRITICAL: Change to DRIVE_ROOT so local imports work
os.chdir(DRIVE_ROOT)
print(f'Working directory changed to: {os.getcwd()}')

In [ ]:
# Cell 3: Imports and helpers
from configs import ALL_METHOD_CONFIGS, DEFAULT_TRAIN_CFG, make_task_cfg, describe_method
from fair_dfl.runner import run_experiment_unified
from fair_dfl.training.method_spec import resolve_method_spec
from plotting import plot_all, plot_gradient_conflict, plot_training_curves
import matplotlib.pyplot as plt

RESULTS_DIR = os.path.join(DRIVE_ROOT, 'results', 'ablation')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Methods to compare in ablation
ABLATION_METHODS = ['FPTO', 'FDFL', 'FFO', 'WS-equal', 'WS-dec', 'MGDA', 'PCGrad']

DATA_CSV = os.path.join(DRIVE_ROOT, 'data', 'data_processed.csv')
ALPHA = 2.0
N_SAMPLE = 1000  # use 0 for full data


def make_summary_table(sdf, group_cols=None):
    """Aggregate test metrics as mean +/- std across seeds.

    Reports **normalized regret** as the primary 'Regret' column (matching
    the main experiment notebooks).  Raw (unnormalized) regret remains in
    the underlying CSV logs.
    """
    if group_cols is None:
        group_cols = ['config_name']

    has_norm = 'test_regret_normalized' in sdf.columns
    regret_col = 'test_regret_normalized' if has_norm else 'test_regret'

    metrics = [regret_col, 'test_fairness', 'test_pred_mse']
    short = {
        'test_regret_normalized': 'Regret',
        'test_regret': 'Regret',
        'test_fairness': 'Fairness',
        'test_pred_mse': 'Pred MSE',
    }
    rows = []
    for keys, grp in sdf.groupby(group_cols):
        if isinstance(keys, str):
            keys = (keys,)
        row = dict(zip(group_cols, keys))
        for m in metrics:
            if m not in grp.columns:
                continue
            mu, sd = grp[m].mean(), grp[m].std()
            row[f'{short[m]}'] = f'{mu:.4f} +/- {sd:.4f}'
            row[f'{short[m]}_mean'] = mu
        row['n_seeds'] = len(grp)
        rows.append(row)
    return pd.DataFrame(rows)


print(f'Methods: {ABLATION_METHODS}')
print(f'Results dir: {RESULTS_DIR}')

---
## Experiment 1: Disable Continuation

By default, FPLG/WS-equal/WS-dec/MGDA/PCGrad/FFO have `continuation=True`,
meaning model weights carry over across lambda stages (0.0 -> 0.5).
FDFL has `continuation=False` and resets to random init each stage.

This explains why gradient cosines start at different values — continuation methods
enter lambda=0.5 already trained for 70 steps, while FDFL starts fresh.

Here we **disable continuation for all methods** to see if they all start the same.

In [ ]:
# Cell 4: Run with continuation=False for all methods

methods_no_cont = {}
for name in ABLATION_METHODS:
    cfg = copy.deepcopy(ALL_METHOD_CONFIGS[name])
    cfg['continuation'] = False  # force reset at each lambda stage
    methods_no_cont[name] = cfg

print('All methods now have continuation=False:')
for name, cfg in methods_no_cont.items():
    spec = resolve_method_spec(cfg)
    print(f'  {name:15s}  continuation={spec.continuation}')

task_cfg = make_task_cfg(data_csv=DATA_CSV, n_sample=N_SAMPLE, alpha_fair=ALPHA)
train_cfg = copy.deepcopy(DEFAULT_TRAIN_CFG)
train_cfg['log_every'] = 1  # log every iteration for detailed cosine tracking

exp_cfg = {
    'task': task_cfg,
    'training': train_cfg,
}

stage_df_nocont, iter_df_nocont = run_experiment_unified(exp_cfg, method_configs=methods_no_cont)
stage_df_nocont['config_name'] = stage_df_nocont['method'].str.upper()
# Map method names back to original config names for plotting
_method_to_config = {name.lower(): name for name in ABLATION_METHODS}
stage_df_nocont['config_name'] = stage_df_nocont['method'].map(
    lambda x: _method_to_config.get(x, x.upper()))
iter_df_nocont['config_name'] = iter_df_nocont['method'].map(
    lambda x: _method_to_config.get(x, x.upper()))
stage_df_nocont['alpha_fair'] = ALPHA
iter_df_nocont['alpha_fair'] = ALPHA
stage_df_nocont['fairness_type'] = 'mad'
iter_df_nocont['fairness_type'] = 'mad'

print(f'\nDone. {len(stage_df_nocont)} stage rows, {len(iter_df_nocont)} iter rows.')

In [ ]:
# Cell 5: Plot gradient cosines — all methods should now start at same point
nocont_dir = os.path.join(RESULTS_DIR, 'no_continuation')
os.makedirs(nocont_dir, exist_ok=True)

plot_gradient_conflict(iter_df_nocont, alpha_values=[ALPHA], results_dir=nocont_dir)
plot_training_curves(iter_df_nocont, alpha_values=[ALPHA], results_dir=nocont_dir)

In [ ]:
# Cell 5b: Exp 1 summary — per method, per lambda (mean +/- std across seeds)

print(f'=== Experiment 1: No Continuation (alpha={ALPHA}) ===')
print(f'Each cell shows mean +/- std across {len(DEFAULT_TRAIN_CFG["seeds"])} seeds')
print(f'Regret = normalized regret (raw regret available in CSV logs)\n')

# One table per lambda value
for lam in sorted(stage_df_nocont['lambda'].unique()):
    sub = stage_df_nocont[stage_df_nocont['lambda'] == lam]
    tbl = make_summary_table(sub)
    tbl = tbl.sort_values('Regret_mean')

    print(f'--- lambda = {lam} ---')
    display_cols = ['config_name', 'Regret', 'Fairness', 'Pred MSE']
    print(tbl[[c for c in display_cols if c in tbl.columns]].to_string(index=False))
    print()

# Best lambda selection per method
has_norm = 'test_regret_normalized' in stage_df_nocont.columns
regret_col = 'test_regret_normalized' if has_norm else 'test_regret'

print('--- Best lambda per method (minimizes normalized regret + fairness) ---')
best_rows = []
for method in sorted(stage_df_nocont['config_name'].unique()):
    msub = stage_df_nocont[stage_df_nocont['config_name'] == method]
    per_lam = msub.groupby('lambda').agg(
        regret=(regret_col, 'mean'),
        fairness=('test_fairness', 'mean'),
        mse=('test_pred_mse', 'mean'),
    ).reset_index()
    if len(per_lam) == 0:
        continue
    r_max = max(per_lam['regret'].max(), 1e-12)
    f_max = max(per_lam['fairness'].max(), 1e-12)
    per_lam['score'] = per_lam['regret'] / r_max + per_lam['fairness'] / f_max
    best = per_lam.loc[per_lam['score'].idxmin()]
    best_rows.append({
        'method': method,
        'best_lambda': best['lambda'],
        'regret': f'{best["regret"]:.4f}',
        'fairness': f'{best["fairness"]:.4f}',
        'mse': f'{best["mse"]:.4f}',
    })

print(pd.DataFrame(best_rows).to_string(index=False))

---
## Experiment 2: Single Lambda = 0.5

Run only lambda=0.5 to eliminate cross-stage effects entirely.
Every method starts from the same random init and runs 70 steps.

In [ ]:
# Cell 6: Run with single lambda=0.5

_method_to_config = {name.lower(): name for name in ABLATION_METHODS}

methods_single_lambda = copy.deepcopy(methods_no_cont)  # also no continuation

train_cfg_single = copy.deepcopy(DEFAULT_TRAIN_CFG)
train_cfg_single['lambdas'] = [0.5]  # single lambda
train_cfg_single['log_every'] = 1

exp_cfg_single = {
    'task': make_task_cfg(data_csv=DATA_CSV, n_sample=N_SAMPLE, alpha_fair=ALPHA),
    'training': train_cfg_single,
}

stage_df_single, iter_df_single = run_experiment_unified(exp_cfg_single, method_configs=methods_single_lambda)
stage_df_single['config_name'] = stage_df_single['method'].map(
    lambda x: _method_to_config.get(x, x.upper()))
iter_df_single['config_name'] = iter_df_single['method'].map(
    lambda x: _method_to_config.get(x, x.upper()))
stage_df_single['alpha_fair'] = ALPHA
iter_df_single['alpha_fair'] = ALPHA
stage_df_single['fairness_type'] = 'mad'
iter_df_single['fairness_type'] = 'mad'

print(f'Done. {len(stage_df_single)} stage rows, {len(iter_df_single)} iter rows.')

In [ ]:
# Cell 7: Plot single-lambda results
single_dir = os.path.join(RESULTS_DIR, 'single_lambda')
os.makedirs(single_dir, exist_ok=True)

plot_gradient_conflict(iter_df_single, alpha_values=[ALPHA], results_dir=single_dir)
plot_training_curves(iter_df_single, alpha_values=[ALPHA], results_dir=single_dir)

In [ ]:
# Cell 7b: Exp 2 summary — single lambda=0.5 (mean +/- std across seeds)

print(f'=== Experiment 2: Single Lambda=0.5 (alpha={ALPHA}) ===')
print(f'Each cell shows mean +/- std across {len(DEFAULT_TRAIN_CFG["seeds"])} seeds')
print(f'Regret = normalized regret (raw regret available in CSV logs)\n')

tbl = make_summary_table(stage_df_single)
tbl = tbl.sort_values('Regret_mean')
tbl.insert(0, 'Rank', range(1, len(tbl) + 1))

display_cols = ['Rank', 'config_name', 'Regret', 'Fairness', 'Pred MSE', 'n_seeds']
print(tbl[[c for c in display_cols if c in tbl.columns]].to_string(index=False))

---
## Experiment 3: Alternative Fairness Measures

Compare the three available prediction-side fairness metrics:
- **MAD** (Mean Absolute Deviation) — default, works for K>=2 groups
- **Gap** (Group Accuracy Parity) — binary group MSE gap
- **Atkinson** (Atkinson Inequality Index) — penalizes extreme disparities

We run the same methods under each fairness metric to see if performance ranking is consistent.

In [ ]:
# Cell 8: Run all three fairness types

_method_to_config = {name.lower(): name for name in ABLATION_METHODS}

fairness_types = ['mad', 'gap', 'atkinson']
results_by_fairness = {}

for ft in fairness_types:
    print(f'\n{"="*60}')
    print(f'Running fairness_type = {ft}')
    print(f'{"="*60}')

    ft_task_cfg = make_task_cfg(
        data_csv=DATA_CSV, n_sample=N_SAMPLE, alpha_fair=ALPHA, fairness_type=ft)
    ft_train_cfg = copy.deepcopy(DEFAULT_TRAIN_CFG)
    ft_train_cfg['lambdas'] = [0.0, 0.5]
    ft_train_cfg['log_every'] = 1

    ft_exp_cfg = {'task': ft_task_cfg, 'training': ft_train_cfg}
    ft_methods = copy.deepcopy(methods_no_cont)

    stage_df_ft, iter_df_ft = run_experiment_unified(ft_exp_cfg, method_configs=ft_methods)
    stage_df_ft['config_name'] = stage_df_ft['method'].map(
        lambda x: _method_to_config.get(x, x.upper()))
    iter_df_ft['config_name'] = iter_df_ft['method'].map(
        lambda x: _method_to_config.get(x, x.upper()))
    stage_df_ft['alpha_fair'] = ALPHA
    iter_df_ft['alpha_fair'] = ALPHA
    stage_df_ft['fairness_type'] = ft
    iter_df_ft['fairness_type'] = ft

    results_by_fairness[ft] = (stage_df_ft, iter_df_ft)
    print(f'{ft}: {len(stage_df_ft)} stage rows, {len(iter_df_ft)} iter rows')

print('\nAll fairness types complete.')

In [ ]:
# Cell 9: Compare fairness measures — per-type tables + rank consistency

print(f'=== Experiment 3: Fairness Measure Comparison (alpha={ALPHA}, lambda=0.5) ===')
print(f'Each cell shows mean +/- std across {len(DEFAULT_TRAIN_CFG["seeds"])} seeds')
print(f'Regret = normalized regret (raw regret available in CSV logs)\n')

# --- One table per fairness type ---
for ft in fairness_types:
    sdf, _ = results_by_fairness[ft]
    tbl = make_summary_table(sdf)
    tbl = tbl.sort_values('Regret_mean')
    tbl.insert(0, 'Rank', range(1, len(tbl) + 1))

    print(f'{"="*80}')
    print(f'  Fairness type: {ft.upper()}')
    print(f'{"="*80}')
    display_cols = ['Rank', 'config_name', 'Regret', 'Fairness', 'Pred MSE']
    print(tbl[[c for c in display_cols if c in tbl.columns]].to_string(index=False))
    print()

# --- Cross-fairness rank comparison ---
print(f'{"="*80}')
print('  Ranking Consistency Across Fairness Types (by normalized regret, 1 = best)')
print(f'{"="*80}')

rank_dict = {}
for ft in fairness_types:
    sdf, _ = results_by_fairness[ft]
    tbl = make_summary_table(sdf).sort_values('Regret_mean').reset_index(drop=True)
    for rank, (_, r) in enumerate(tbl.iterrows(), 1):
        method = r['config_name']
        if method not in rank_dict:
            rank_dict[method] = {}
        rank_dict[method][ft.upper()] = rank

rank_table = pd.DataFrame.from_dict(rank_dict, orient='index')
rank_table.index.name = 'method'
rank_table['Avg Rank'] = rank_table.mean(axis=1)
rank_table = rank_table.sort_values('Avg Rank')
print(rank_table.to_string(float_format='%.1f'))

# --- Regret ranking order per fairness type ---
print('\n--- Regret ordering (best -> worst) ---')
for ft in fairness_types:
    sdf, _ = results_by_fairness[ft]
    tbl = make_summary_table(sdf).sort_values('Regret_mean')
    ranking = list(tbl['config_name'].values)
    print(f'  {ft.upper():10s}: {" > ".join(ranking)}')

In [ ]:
# Cell 10: Side-by-side gradient conflict plots for each fairness type
for ft, (sdf, idf) in results_by_fairness.items():
    ft_dir = os.path.join(RESULTS_DIR, f'fairness_{ft}')
    os.makedirs(ft_dir, exist_ok=True)
    print(f'\n--- Gradient conflict: {ft} ---')
    plot_gradient_conflict(idf, alpha_values=[ALPHA], results_dir=ft_dir)
    plot_training_curves(idf, alpha_values=[ALPHA], results_dir=ft_dir)

---
## Gradient Cosine Similarity Table

Print per-iteration gradient cosine values as a table for detailed inspection.
Uses the single-lambda experiment (Experiment 2) data.

In [ ]:
# Cell 11: Gradient cosine table — per method, per iteration

cos_cols = ['cos_dec_pred', 'cos_dec_fair', 'cos_pred_fair']
norm_cols = ['grad_norm_dec', 'grad_norm_pred', 'grad_norm_fair', 'grad_norm_combined']
display_cols = ['iter'] + cos_cols + norm_cols

# Use single-lambda data (iter_df_single)
df = iter_df_single.copy()

# Average across seeds
for method in sorted(df['config_name'].unique()):
    msub = df[df['config_name'] == method]
    available = [c for c in display_cols if c in msub.columns]
    grouped = msub.groupby('iter')[available[1:]].mean().reset_index()
    grouped.insert(0, 'iter', grouped.index if 'iter' not in grouped.columns else grouped['iter'])

    print(f'\n{"="*100}')
    print(f'Method: {method}  |  lambda=0.5  |  alpha_fair={ALPHA}  |  averaged over seeds')
    print(f'{"="*100}')

    # Format for readability
    fmt_df = grouped.copy()
    for c in cos_cols:
        if c in fmt_df.columns:
            fmt_df[c] = fmt_df[c].map(lambda x: f'{x:+.4f}')
    for c in norm_cols:
        if c in fmt_df.columns:
            fmt_df[c] = fmt_df[c].map(lambda x: f'{x:.6f}')

    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 200):
        print(fmt_df[[c for c in display_cols if c in fmt_df.columns]].to_string(index=False))

In [ ]:
# Cell 12: Summary comparison table — iteration 0 vs final iteration

df = iter_df_single.copy()
compare_rows = []

for method in sorted(df['config_name'].unique()):
    msub = df[df['config_name'] == method]
    iters = sorted(msub['iter'].unique())
    if len(iters) < 2:
        continue
    first_iter, last_iter = iters[0], iters[-1]

    for label, it in [('start', first_iter), ('end', last_iter)]:
        isub = msub[msub['iter'] == it]
        row = {'method': method, 'phase': label, 'iter': it}
        for c in cos_cols + norm_cols + ['loss_dec', 'loss_pred', 'loss_fair']:
            if c in isub.columns:
                row[c] = isub[c].mean()
        compare_rows.append(row)

compare_df = pd.DataFrame(compare_rows)
print('=== Start vs End comparison (single lambda=0.5, averaged over seeds) ===')
print()

# Display start rows
start = compare_df[compare_df['phase'] == 'start'].sort_values('method')
end = compare_df[compare_df['phase'] == 'end'].sort_values('method')

print('--- Iteration 0 (all methods should be identical if same init) ---')
show_cols = ['method'] + [c for c in cos_cols + ['loss_dec', 'loss_pred', 'loss_fair'] if c in start.columns]
print(start[show_cols].to_string(index=False, float_format='%.4f'))

print(f'\n--- Final iteration ---')
print(end[show_cols].to_string(index=False, float_format='%.4f'))

In [ ]:
# Cell 13: Save all ablation data to CSV

stage_df_nocont.to_csv(os.path.join(RESULTS_DIR, 'stage_no_continuation.csv'), index=False)
iter_df_nocont.to_csv(os.path.join(RESULTS_DIR, 'iter_no_continuation.csv'), index=False)

stage_df_single.to_csv(os.path.join(RESULTS_DIR, 'stage_single_lambda.csv'), index=False)
iter_df_single.to_csv(os.path.join(RESULTS_DIR, 'iter_single_lambda.csv'), index=False)

for ft, (sdf, idf) in results_by_fairness.items():
    sdf.to_csv(os.path.join(RESULTS_DIR, f'stage_fairness_{ft}.csv'), index=False)
    idf.to_csv(os.path.join(RESULTS_DIR, f'iter_fairness_{ft}.csv'), index=False)

print(f'All results saved to {RESULTS_DIR}/')
!ls -la {RESULTS_DIR}/

---
## Unified Cross-Fairness Comparison

Compares all methods across fairness types using **Relative Fairness** =
fairness(λ=0.5) / fairness(λ=0), which normalizes GAP (~0-200), MAD (~0-100),
and Atkinson (0-1) to a common "fraction of baseline unfairness remaining" scale.

Uses Exp 3 data only (all three fairness types under identical conditions).

In [ ]:
# Cell 14: Unified cross-fairness comparison table

import warnings

# --- Step 1: Collect Exp 3 stage DataFrames only ---
all_stage_dfs = []
for ft, (sdf, _) in results_by_fairness.items():
    df = sdf.copy()
    if 'fairness_type' not in df.columns:
        df['fairness_type'] = ft
    all_stage_dfs.append(df)

if not all_stage_dfs:
    print('No Exp 3 data available for unified comparison.')
else:
    combined = pd.concat(all_stage_dfs, ignore_index=True)

    # --- Step 2: Compute relative fairness per (method, fairness_type, seed) ---
    has_norm = 'test_regret_normalized' in combined.columns
    regret_col = 'test_regret_normalized' if has_norm else 'test_regret'

    rows = []
    for (method, ft), grp in combined.groupby(['config_name', 'fairness_type']):
        lam0 = grp[grp['lambda'] == 0.0]
        lam05 = grp[grp['lambda'] == 0.5]

        if lam0.empty or lam05.empty:
            warnings.warn(f'Missing lambda=0 or lambda=0.5 for {method}/{ft}, skipping.')
            continue

        for seed in lam05['seed'].unique():
            s0 = lam0[lam0['seed'] == seed]
            s05 = lam05[lam05['seed'] == seed]
            if s0.empty or s05.empty:
                continue

            baseline = s0['test_fairness'].values[0]
            rel_fair = s05['test_fairness'].values[0] / baseline if baseline > 1e-12 else float('nan')

            rows.append({
                'method': method, 'fairness_type': ft, 'seed': seed,
                'norm_regret': s05[regret_col].values[0],
                'relative_fairness': rel_fair,
                'pred_mse': s05['test_pred_mse'].values[0],
            })

    if not rows:
        print('Could not compute relative fairness — missing lambda=0 baseline.')
        print('Re-run Exp 3 with lambdas=[0.0, 0.5] to generate baselines.')
    else:
        detail_df = pd.DataFrame(rows)

        # --- Step 3: Aggregate mean +/- std across seeds ---
        agg = detail_df.groupby(['method', 'fairness_type']).agg(
            norm_regret_mean=('norm_regret', 'mean'),
            norm_regret_std=('norm_regret', 'std'),
            rel_fair_mean=('relative_fairness', 'mean'),
            rel_fair_std=('relative_fairness', 'std'),
            pred_mse_mean=('pred_mse', 'mean'),
            pred_mse_std=('pred_mse', 'std'),
            n_seeds=('seed', 'count'),
        ).reset_index()

        agg['Norm Regret'] = agg.apply(
            lambda r: f"{r['norm_regret_mean']:.4f} +/- {r['norm_regret_std']:.4f}", axis=1)
        agg['Rel. Fairness'] = agg.apply(
            lambda r: f"{r['rel_fair_mean']:.4f} +/- {r['rel_fair_std']:.4f}"
            if not np.isnan(r['rel_fair_mean']) else 'N/A', axis=1)
        agg['Pred MSE'] = agg.apply(
            lambda r: f"{r['pred_mse_mean']:.4f} +/- {r['pred_mse_std']:.4f}", axis=1)

        print('=' * 100)
        print(f'UNIFIED CROSS-FAIRNESS COMPARISON (alpha={ALPHA}, lambda=0.5)')
        print('Relative Fairness = fairness(lam=0.5) / fairness(lam=0)  [lower = more improvement]')
        print('=' * 100)

        display_cols = ['method', 'fairness_type', 'Norm Regret', 'Rel. Fairness', 'Pred MSE']
        print(agg[display_cols].sort_values(['fairness_type', 'norm_regret_mean']).to_string(index=False))

        # --- Step 4: Rank consistency by normalized regret ---
        print(f'\n{"=" * 100}')
        print('RANK CONSISTENCY (by norm_regret, 1 = best)')
        print(f'{"=" * 100}')

        rank_dict = {}
        for ft, ft_grp in agg.groupby('fairness_type'):
            ft_sorted = ft_grp.sort_values('norm_regret_mean').reset_index(drop=True)
            for rank, (_, r) in enumerate(ft_sorted.iterrows(), 1):
                rank_dict.setdefault(r['method'], {})[ft.upper()] = rank

        rank_table = pd.DataFrame.from_dict(rank_dict, orient='index')
        rank_table.index.name = 'Method'
        rank_table['Avg Rank'] = rank_table.mean(axis=1)
        rank_table = rank_table.sort_values('Avg Rank')
        print(rank_table.to_string(float_format='%.1f'))

        # --- Step 5: Rank consistency by relative fairness ---
        print(f'\n{"=" * 100}')
        print('RANK CONSISTENCY (by relative_fairness, 1 = most fairness improvement)')
        print(f'{"=" * 100}')

        fair_rank_dict = {}
        for ft, ft_grp in agg.groupby('fairness_type'):
            ft_sorted = ft_grp.sort_values('rel_fair_mean').reset_index(drop=True)
            for rank, (_, r) in enumerate(ft_sorted.iterrows(), 1):
                fair_rank_dict.setdefault(r['method'], {})[ft.upper()] = rank

        fair_rank_table = pd.DataFrame.from_dict(fair_rank_dict, orient='index')
        fair_rank_table.index.name = 'Method'
        fair_rank_table['Avg Rank'] = fair_rank_table.mean(axis=1)
        fair_rank_table = fair_rank_table.sort_values('Avg Rank')
        print(fair_rank_table.to_string(float_format='%.1f'))

---
## Experiment 4: Predictor Architecture

Compare five predictor architectures while keeping all other settings identical
(continuation=False, lambda=0.5, alpha=2.0):

| Architecture | Description |
|---|---|
| `linear` | Single linear layer (no hidden layers) |
| `mlp_64x2` | 2-layer MLP, 64 hidden (current default) |
| `mlp_128x3` | 3-layer MLP, 128 hidden |
| `resnet` | ResNet-Tabular (3 residual blocks, 128 hidden, dropout=0.1) |
| `ft_transformer` | FT-Transformer (2 layers, 4 heads, d_token=64) |

In [ ]:
# Cell 15: Run predictor architecture comparison

_method_to_config = {name.lower(): name for name in ABLATION_METHODS}

ARCHITECTURE_CONFIGS = {
    'linear': {'arch': 'linear'},
    'mlp_64x2': {'arch': 'mlp', 'hidden_dim': 64, 'n_layers': 2},
    'mlp_128x3': {'arch': 'mlp', 'hidden_dim': 128, 'n_layers': 3},
    'resnet': {'arch': 'resnet_tabular', 'hidden_dim': 128, 'n_blocks': 3, 'dropout': 0.1},
    'ft_transformer': {'arch': 'ft_transformer', 'd_token': 64, 'n_heads': 4, 'n_layers': 2, 'dropout': 0.1},
}

results_arch = {}
for arch_name, model_cfg in ARCHITECTURE_CONFIGS.items():
    print(f'\n{"="*60}')
    print(f'Architecture: {arch_name}')
    print(f'{"="*60}')

    arch_train_cfg = copy.deepcopy(DEFAULT_TRAIN_CFG)
    arch_train_cfg['lambdas'] = [0.5]
    arch_train_cfg['log_every'] = 1
    arch_train_cfg['model'] = model_cfg

    arch_exp_cfg = {
        'task': make_task_cfg(data_csv=DATA_CSV, n_sample=N_SAMPLE, alpha_fair=ALPHA),
        'training': arch_train_cfg,
    }
    arch_methods = copy.deepcopy(methods_no_cont)

    sdf, idf = run_experiment_unified(arch_exp_cfg, method_configs=arch_methods)
    sdf['config_name'] = sdf['method'].map(lambda x: _method_to_config.get(x, x.upper()))
    idf['config_name'] = idf['method'].map(lambda x: _method_to_config.get(x, x.upper()))
    sdf['alpha_fair'] = ALPHA
    sdf['hp_config'] = arch_name
    idf['hp_config'] = arch_name
    results_arch[arch_name] = (sdf, idf)
    print(f'{arch_name}: {len(sdf)} stage rows')

print('\nAll architectures complete.')

In [ ]:
# Cell 16: Exp 4 summary — architecture comparison

print(f'=== Experiment 4: Predictor Architecture (alpha={ALPHA}, lambda=0.5) ===')
print(f'Regret = normalized regret\n')

for arch_name, (sdf, _) in results_arch.items():
    tbl = make_summary_table(sdf)
    tbl = tbl.sort_values('Regret_mean')
    tbl.insert(0, 'Rank', range(1, len(tbl) + 1))
    print(f'--- {arch_name} ---')
    display_cols = ['Rank', 'config_name', 'Regret', 'Fairness', 'Pred MSE']
    print(tbl[[c for c in display_cols if c in tbl.columns]].to_string(index=False))
    print()

# Rank consistency across architectures
print(f'{"="*80}')
print('Ranking Consistency Across Architectures (by norm regret, 1 = best)')
print(f'{"="*80}')

rank_dict = {}
for arch_name, (sdf, _) in results_arch.items():
    tbl = make_summary_table(sdf).sort_values('Regret_mean').reset_index(drop=True)
    for rank, (_, r) in enumerate(tbl.iterrows(), 1):
        rank_dict.setdefault(r['config_name'], {})[arch_name] = rank

rank_table = pd.DataFrame.from_dict(rank_dict, orient='index')
rank_table.index.name = 'Method'
rank_table['Avg Rank'] = rank_table.mean(axis=1)
rank_table = rank_table.sort_values('Avg Rank')
print(rank_table.to_string(float_format='%.1f'))

---
## Experiment 5: Learning Rate × Steps Grid

Sweep over learning rate {0.0001, 0.0005, 0.001, 0.005} and
steps_per_lambda {50, 100, 200} to test convergence sensitivity.
All other settings: continuation=False, lambda=0.5, SGD optimizer.

In [ ]:
# Cell 17: Run LR x Steps grid

_method_to_config = {name.lower(): name for name in ABLATION_METHODS}

LR_STEPS_GRID = [
    {'lr': lr, 'steps_per_lambda': steps}
    for lr in [0.0001, 0.0005, 0.001, 0.005]
    for steps in [50, 100, 200]
]

results_lr_steps = {}
for hp in LR_STEPS_GRID:
    hp_label = f"lr={hp['lr']}_steps={hp['steps_per_lambda']}"
    print(f'\n--- {hp_label} ---')

    ls_train_cfg = copy.deepcopy(DEFAULT_TRAIN_CFG)
    ls_train_cfg['lambdas'] = [0.5]
    ls_train_cfg['log_every'] = 1
    ls_train_cfg['lr'] = hp['lr']
    ls_train_cfg['steps_per_lambda'] = hp['steps_per_lambda']

    ls_exp_cfg = {
        'task': make_task_cfg(data_csv=DATA_CSV, n_sample=N_SAMPLE, alpha_fair=ALPHA),
        'training': ls_train_cfg,
    }
    ls_methods = copy.deepcopy(methods_no_cont)

    sdf, idf = run_experiment_unified(ls_exp_cfg, method_configs=ls_methods)
    sdf['config_name'] = sdf['method'].map(lambda x: _method_to_config.get(x, x.upper()))
    sdf['alpha_fair'] = ALPHA
    sdf['hp_config'] = hp_label
    results_lr_steps[hp_label] = (sdf, idf)
    print(f'{hp_label}: {len(sdf)} stage rows')

print('\nLR x Steps grid complete.')

In [ ]:
# Cell 18: Exp 5 summary — LR x Steps

print(f'=== Experiment 5: LR x Steps (alpha={ALPHA}, lambda=0.5) ===')
print(f'Regret = normalized regret\n')

# Collect per-method average across HP configs
all_hp_rows = []
for hp_label, (sdf, _) in results_lr_steps.items():
    tbl = make_summary_table(sdf)
    tbl['hp_config'] = hp_label
    all_hp_rows.append(tbl)

if all_hp_rows:
    hp_df = pd.concat(all_hp_rows, ignore_index=True)

    # Show best HP per method
    print('--- Best HP config per method (lowest norm regret) ---')
    for method in sorted(hp_df['config_name'].unique()):
        msub = hp_df[hp_df['config_name'] == method]
        best = msub.loc[msub['Regret_mean'].idxmin()]
        print(f'  {method:15s}  best={best["hp_config"]:30s}  regret={best["Regret"]}')

    # Rank consistency across HP configs
    print(f'\n{"="*80}')
    print('Method ranking stability across LR x Steps configs')
    print(f'{"="*80}')

    method_ranks = {}
    for hp_label, (sdf, _) in results_lr_steps.items():
        tbl = make_summary_table(sdf).sort_values('Regret_mean').reset_index(drop=True)
        for rank, (_, r) in enumerate(tbl.iterrows(), 1):
            method_ranks.setdefault(r['config_name'], []).append(rank)

    stability = []
    for method, ranks in sorted(method_ranks.items()):
        stability.append({
            'method': method,
            'mean_rank': np.mean(ranks),
            'std_rank': np.std(ranks),
            'min_rank': min(ranks),
            'max_rank': max(ranks),
        })
    stab_df = pd.DataFrame(stability).sort_values('mean_rank')
    print(stab_df.to_string(index=False, float_format='%.2f'))

---
## Experiment 6: Optimizer (SGD vs Adam)

Compare SGD (current default) with Adam optimizer.
All other settings: continuation=False, lambda=0.5, default LR and steps.

In [ ]:
# Cell 19: Run optimizer comparison

_method_to_config = {name.lower(): name for name in ABLATION_METHODS}

OPTIMIZERS = ['sgd', 'adam']
results_optimizer = {}

for opt in OPTIMIZERS:
    print(f'\n{"="*60}')
    print(f'Optimizer: {opt.upper()}')
    print(f'{"="*60}')

    opt_train_cfg = copy.deepcopy(DEFAULT_TRAIN_CFG)
    opt_train_cfg['lambdas'] = [0.5]
    opt_train_cfg['log_every'] = 1
    opt_train_cfg['optimizer'] = opt

    opt_exp_cfg = {
        'task': make_task_cfg(data_csv=DATA_CSV, n_sample=N_SAMPLE, alpha_fair=ALPHA),
        'training': opt_train_cfg,
    }
    opt_methods = copy.deepcopy(methods_no_cont)

    sdf, idf = run_experiment_unified(opt_exp_cfg, method_configs=opt_methods)
    sdf['config_name'] = sdf['method'].map(lambda x: _method_to_config.get(x, x.upper()))
    sdf['alpha_fair'] = ALPHA
    sdf['hp_config'] = opt
    results_optimizer[opt] = (sdf, idf)
    print(f'{opt}: {len(sdf)} stage rows')

print('\nOptimizer comparison complete.')

In [ ]:
# Cell 20: Exp 6 summary — optimizer comparison

print(f'=== Experiment 6: Optimizer Comparison (alpha={ALPHA}, lambda=0.5) ===')
print(f'Regret = normalized regret\n')

for opt, (sdf, _) in results_optimizer.items():
    tbl = make_summary_table(sdf)
    tbl = tbl.sort_values('Regret_mean')
    tbl.insert(0, 'Rank', range(1, len(tbl) + 1))
    print(f'--- {opt.upper()} ---')
    display_cols = ['Rank', 'config_name', 'Regret', 'Fairness', 'Pred MSE']
    print(tbl[[c for c in display_cols if c in tbl.columns]].to_string(index=False))
    print()

# Side-by-side comparison
print(f'{"="*80}')
print('SGD vs Adam — per method delta (Adam - SGD)')
print(f'{"="*80}')

if 'sgd' in results_optimizer and 'adam' in results_optimizer:
    sgd_tbl = make_summary_table(results_optimizer['sgd'][0]).set_index('config_name')
    adam_tbl = make_summary_table(results_optimizer['adam'][0]).set_index('config_name')
    common = sorted(set(sgd_tbl.index) & set(adam_tbl.index))
    for method in common:
        sgd_r = sgd_tbl.loc[method, 'Regret_mean']
        adam_r = adam_tbl.loc[method, 'Regret_mean']
        delta = adam_r - sgd_r
        winner = 'Adam' if delta < 0 else 'SGD'
        print(f'  {method:15s}  SGD={sgd_r:.4f}  Adam={adam_r:.4f}  delta={delta:+.4f}  winner={winner}')

In [ ]:
# Cell 21: Save HP ablation results to CSV

# Architecture
arch_dfs = [sdf for sdf, _ in results_arch.values()]
if arch_dfs:
    pd.concat(arch_dfs, ignore_index=True).to_csv(
        os.path.join(RESULTS_DIR, 'stage_hp_architecture.csv'), index=False)

# LR x Steps
ls_dfs = [sdf for sdf, _ in results_lr_steps.values()]
if ls_dfs:
    pd.concat(ls_dfs, ignore_index=True).to_csv(
        os.path.join(RESULTS_DIR, 'stage_hp_lr_steps.csv'), index=False)

# Optimizer
opt_dfs = [sdf for sdf, _ in results_optimizer.values()]
if opt_dfs:
    pd.concat(opt_dfs, ignore_index=True).to_csv(
        os.path.join(RESULTS_DIR, 'stage_hp_optimizer.csv'), index=False)

print(f'HP ablation results saved to {RESULTS_DIR}/')
!ls -la {RESULTS_DIR}/